<a href="https://colab.research.google.com/github/anitaKiran/lab-extractive-question-answering/blob/main/lab_extractive_question_answering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LAB | Extractive Question Answering

<div style="border:1px solid #ccc; border-radius:6px; text-align: center;">

<br>

**Run this notebook in [Google Colab](https://colab.research.google.com/) for GPU acceleration.**

<br>

</div>

This notebook demonstrates how Pinecone helps you build an extractive question-answering application. To build an extractive question-answering system, we need three main components:

- A vector index to store and run semantic search
- A retriever model for embedding context passages
- A reader model to extract answers

We will use the SQuAD dataset, which consists of **questions** and **context** paragraphs containing question **answers**. We generate embeddings for the context passages using the retriever, index them in the vector database, and query with semantic search to retrieve the top k most relevant contexts containing potential answers to our question. We then use the reader model to extract the answers from the returned contexts.

Let's get started by installing the packages needed for notebook to run:

# Install Dependencies

In [1]:
!pip install -q \
    transformers==4.46.3 \
    datasets==4.5.0 \
    pinecone==5.4.2 \
    sentence-transformers==3.3.1 \
    huggingface_hub==0.26.5

In [ ]:
# Force-kill the kernel process to restart it, ensuring the newly
# installed/uninstalled package versions are loaded in a fresh process

import os
import sys

print(
    "⚠️ WARNING:\n"
    "This code cell will restart the kernel to load the newly installed packages...\n"
    "If you used 'Run all', it will stop here — that's expected.\n"
    "See the note below on how to continue running the rest of the notebook.",
    flush=True,
)
sys.stdout.flush()

os.kill(os.getpid(), 9)

⚠️ WARNING:
This code cell will restart the kernel to load the newly installed packages...
If you used 'Run all', it will stop here — that's expected.
See the note below on how to continue running the rest of the notebook.


<div style="border:1px solid #ccc; border-radius:6px; padding:12px;">

⚠️ **Do not use "Run all" for this whole notebook in one go.** The cell above force-restarts the kernel (`os.kill`) so the newly installed packages load cleanly. Colab's "Run all" cannot continue past a kernel restart — it will always stop right after that cell, no matter how many times you retry it.

Instead, run it in two steps:
1. Run only the two cells above (install + restart). Wait for the kernel to finish restarting (a few seconds).
2. Click on the next cell below, then use **Runtime → Run after** (or manually run the remaining cells one by one / select them and press Shift+Enter). Do not click "Run all" again, since it would just re-trigger the restart at the same spot.

</div>

In [6]:
import transformers
import datasets
import pinecone
import sentence_transformers

print("All required packages imported successfully!")

All required packages imported successfully!


# Load Dataset

Now let's load the SQUAD dataset from the HuggingFace Model Hub. We load the dataset into a pandas dataframe and filter the title, question, and context columns, and we drop any duplicate context passages.

In [8]:
from datasets import load_dataset

# load the squad dataset into a pandas dataframe
df = load_dataset("rajpurkar/squad", split="train").to_pandas()

In [9]:
# Select only the title and context columns
df = df[["title", "context"]]

# Drop rows with duplicate context passages
df = df.drop_duplicates(subset="context").reset_index(drop=True)

# Display the prepared dataset
df

,title,context
0,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha..."
1,University_of_Notre_Dame,"As at most other universities, Notre Dame's st..."
2,University_of_Notre_Dame,The university is the major seat of the Congre...
3,University_of_Notre_Dame,The College of Engineering was established in ...
4,University_of_Notre_Dame,All of Notre Dame's undergraduate students are...
...,...,...
18886,Kathmandu,"Institute of Medicine, the central college of ..."
18887,Kathmandu,Football and Cricket are the most popular spor...
18888,Kathmandu,The total length of roads in Nepal is recorded...
18889,Kathmandu,The main international airport serving Kathman...


# Pinecone Setup

Pinecone is a managed vector database that lets us store context embeddings and run fast semantic search over them. In this section, we set up the connection to Pinecone using an API key and create the index that will hold our context embeddings.

In [1]:
import os
import getpass
from pinecone import Pinecone

#
# For this notebook, you'll need a Pinecone API key,
# to get one, sign up for free at https://app.pinecone.io/,
# then go to "API Keys" and copy the default key (or create a new one).
#
# Once you have your Pinecone API key,
# load it from Colab's "Secrets" manager
# (the key icon in the left sidebar).
# Add a secret named PINECONE_API_KEY there and enable notebook access.
#
# Note: Colab Secrets rely on a UI popup to grant access, which only works when
# running in the Colab UI itself. If you're running this notebook through
# another editor (e.g. VS Code's Colab extension), that popup isn't available,
# so we fall back to an environment variable or a secure manual prompt instead.
api_key = os.environ.get('PINECONE_API_KEY')


if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('PINECONE_API_KEY')
    except Exception:
        api_key = None

if not api_key:
    api_key = getpass.getpass('Enter your Pinecone API key: ')



<br>

Once we have the API key, we can initialize the connection:

In [2]:
from pinecone import Pinecone, ServerlessSpec

spec = ServerlessSpec(
    cloud="aws", region="us-east-1"
)

# connect to pinecone environment
pc = Pinecone(
    api_key = api_key
)

Now we create a new index called "question-answering" — we can name the index anything we want. We specify the metric type as "cosine" and dimension as 384 because the retriever we use to generate context embeddings is optimized for cosine similarity and outputs 384-dimension vectors.

In [3]:
import time

index_name = "question-answering"

# Create the index only if it doesn't already exist
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=spec
    )

# Wait until Pinecone reports that the index is ready
while not pc.describe_index(index_name).status["ready"]:
    print("Waiting for the index to be ready...")
    time.sleep(5)

# Connect to the index
index = pc.Index(index_name)

print(f"Index '{index_name}' is ready!")

Index 'question-answering' is ready!


# Initialize Retriever

Next, we need to initialize our retriever. The retriever will mainly do two things:

- Generate embeddings for all context passages (context vectors/embeddings)
- Generate embeddings for our questions (query vector/embedding)

The retriever will generate embeddings in a way that the questions and context passages containing answers to our questions are nearby in the vector space. We can use cosine similarity to calculate the similarity between the query and context embeddings to find the context passages that contain potential answers to our question.

We will use a SentenceTransformer model named ``multi-qa-MiniLM-L6-cos-v1`` designed for semantic search and trained on 215M (question, answer) pairs from diverse sources as our retriever.

In [4]:
import torch
from sentence_transformers import SentenceTransformer

# Use the GPU if available; otherwise, use the CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Load the retriever model
retriever = SentenceTransformer(
    "multi-qa-MiniLM-L6-cos-v1",
    device=device
)

retriever

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Using device: cuda


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

# Generate Embeddings and Upsert

Next, we need to generate embeddings for the context passages. We will do this in batches to help us more quickly generate embeddings and upload them to the Pinecone index. When passing the documents to Pinecone, we need an id (a unique value), context embedding, and metadata for each document representing context passages in the dataset. The metadata is a dictionary containing data relevant to our embeddings, such as the article title, context passage, etc.

In [10]:
from tqdm.auto import tqdm

# Process 64 paragraphs at a time
batch_size = 64

for i in tqdm(range(0, len(df), batch_size)):
    # Select a batch of paragraphs
    batch = df.iloc[i:i + batch_size]

    # Create embeddings for the paragraphs
    embeddings = retriever.encode(
        batch["context"].tolist(),
        show_progress_bar=False
    )

    # Prepare vectors for Pinecone
    vectors = []

    for row_number, (_, row) in enumerate(batch.iterrows()):
        vectors.append({
            "id": str(i + row_number),
            "values": embeddings[row_number].tolist(),
            "metadata": {
                "title": row["title"],
                "context": row["context"]
            }
        })

    # Upload the batch to Pinecone
    index.upsert(vectors=vectors)

print("Upload complete!")
print(index.describe_index_stats())


  0%|          | 0/296 [00:00<?, ?it/s]

Upload complete!
{'dimension': 384,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 18891}},
 'total_vector_count': 18891}


# Initialize Reader

We use the `deepset/electra-base-squad2` model from the HuggingFace model hub as our reader model. We load this model into a "question-answering" pipeline from HuggingFace transformers and feed it our questions and context passages individually. The model gives a prediction for each context we pass through the pipeline.

In [11]:
from transformers import pipeline

model_name = 'deepset/electra-base-squad2'
# load the reader model into a question-answering pipeline

device_id = 0 if torch.cuda.is_available() else -1
reader = pipeline(tokenizer=model_name, model=model_name, task='question-answering', device=device_id)

reader

config.json:   0%|          | 0.00/635 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Now all the components we need are ready. Let's write some helper functions to execute our queries. The `get_context` function retrieves the context embeddings containing answers to our question from the Pinecone index, and the `extract_answer` function extracts the answers from these context passages.

In [15]:
# Retrieve relevant context passages from Pinecone
def get_context(question, top_k):
    # Convert the question into an embedding
    xq = retriever.encode(question).tolist()

    # Search the index for similar paragraph embeddings
    xc = index.query(
        vector=xq,
        top_k=top_k,
        include_metadata=True
    )

    # Collect the original context text from each match
    contexts = [match["metadata"]["context"] for match in xc["matches"]]

    return contexts

In [16]:
from pprint import pprint

# extracts answer from the context passage
def extract_answer(question, context):
    results = []
    for c in context:
        # feed the reader the question and contexts to extract answers
        answer = reader(question=question, context=c)
        # add the context to answer dict for printing both together
        answer["context"] = c
        results.append(answer)
    # sort the result based on the score from reader model
    sorted_result = pprint(sorted(results, key=lambda x: x['score'], reverse=True))
    return sorted_result

In [17]:
question = "How much oil is Egypt producing in a day?"
context = get_context(question, top_k = 1)
context

['Egypt was producing 691,000 bbl/d of oil and 2,141.05 Tcf of natural gas (in 2013), which makes Egypt as the largest oil producer not member of the Organization of the Petroleum Exporting Countries (OPEC) and the second-largest dry natural gas producer in Africa. In 2013, Egypt was the largest consumer of oil and natural gas in Africa, as more than 20% of total oil consumption and more than 40% of total dry natural gas consumption in Africa. Also, Egypt possesses the largest oil refinery capacity in Africa 726,000 bbl/d (in 2012). Egypt is currently planning to build its first nuclear power plant in El Dabaa city, northern Egypt.']

As we can see, the retiever is working fine and gets us the context passage that contains the answer to our question. Now let's use the reader to extract the exact answer from the context passage.

In [18]:
extract_answer(question, context)

[{'answer': '691,000 bbl/d',
  'context': 'Egypt was producing 691,000 bbl/d of oil and 2,141.05 Tcf of '
             'natural gas (in 2013), which makes Egypt as the largest oil '
             'producer not member of the Organization of the Petroleum '
             'Exporting Countries (OPEC) and the second-largest dry natural '
             'gas producer in Africa. In 2013, Egypt was the largest consumer '
             'of oil and natural gas in Africa, as more than 20% of total oil '
             'consumption and more than 40% of total dry natural gas '
             'consumption in Africa. Also, Egypt possesses the largest oil '
             'refinery capacity in Africa 726,000 bbl/d (in 2012). Egypt is '
             'currently planning to build its first nuclear power plant in El '
             'Dabaa city, northern Egypt.',
  'end': 33,
  'score': 0.9999852180480957,
  'start': 20}]


The reader model predicted with 99% accuracy the correct answer *691,000 bbl/d* as seen from the context passage. Let's run few more queries.

In [19]:
question = "What are the first names of the men that invented youtube?"
context = get_context(question, top_k=1)
extract_answer(question, context)

[{'answer': 'Hurley and Chen',
  'context': 'According to a story that has often been repeated in the media, '
             'Hurley and Chen developed the idea for YouTube during the early '
             'months of 2005, after they had experienced difficulty sharing '
             "videos that had been shot at a dinner party at Chen's apartment "
             'in San Francisco. Karim did not attend the party and denied that '
             'it had occurred, but Chen commented that the idea that YouTube '
             'was founded after a dinner party "was probably very strengthened '
             'by marketing ideas around creating a story that was very '
             'digestible".',
  'end': 79,
  'score': 0.9999276399612427,
  'start': 64}]


In [20]:
question = "What is Albert Eistein famous for?"
context = get_context(question, top_k=1)
extract_answer(question, context)

[{'answer': 'his theories of special relativity and general relativity',
  'context': 'Albert Einstein is known for his theories of special relativity '
             'and general relativity. He also made important contributions to '
             'statistical mechanics, especially his mathematical treatment of '
             'Brownian motion, his resolution of the paradox of specific '
             'heats, and his connection of fluctuations and dissipation. '
             'Despite his reservations about its interpretation, Einstein also '
             'made contributions to quantum mechanics and, indirectly, quantum '
             'field theory, primarily through his theoretical studies of the '
             'photon.',
  'end': 86,
  'score': 0.9500371217727661,
  'start': 29}]


Let's run another question. This time for top 3 context passages from the retriever.

In [21]:
question = "Who was the first person to step foot on the moon?"
context = get_context(question, top_k=3)
extract_answer(question, context)

[{'answer': 'Armstrong',
  'context': 'The trip to the Moon took just over three days. After achieving '
             'orbit, Armstrong and Aldrin transferred into the Lunar Module, '
             'named Eagle, and after a landing gear inspection by Collins '
             'remaining in the Command/Service Module Columbia, began their '
             'descent. After overcoming several computer overload alarms '
             'caused by an antenna switch left in the wrong position, and a '
             'slight downrange error, Armstrong took over manual flight '
             'control at about 180 meters (590 ft), and guided the Lunar '
             'Module to a safe landing spot at 20:18:04 UTC, July 20, 1969 '
             '(3:17:04 pm CDT). The first humans on the Moon would wait '
             'another six hours before they ventured out of their craft. At '
             '02:56 UTC, July 21 (9:56 pm CDT July 20), Armstrong became the '
             'first human to set foot on the Moon.',

<br>

### 🚀 Challenge

Try adding a few more questions of your own in new cells below (reusing `get_context` and `extract_answer`). Experiment with different `top_k` values and observe:

- Does the retriever always return a context that actually contains the answer?
- How does the reader's confidence score change with harder or more ambiguous questions?
- What happens with questions that have no good answer in the dataset?

In [23]:
# ...
# ...
# ...

# Test questions and different numbers of retrieved paragraphs
experiments = [
    {
        "question": "Who was the first person to walk on the Moon?",
        "top_k": 1
    },
    {
        "question": "Who was the first person to walk on the Moon?",
        "top_k": 3
    },
    {
        "question": "What university did Albert Einstein attend in Switzerland?",
        "top_k": 3
    },
    {
        "question": "What is the name of the first hotel built on Mars?",
        "top_k": 3
    }
]

for experiment in experiments:
    question = experiment["question"]
    top_k = experiment["top_k"]

    print("\n" + "=" * 80)
    print(f"QUESTION: {question}")
    print(f"TOP K: {top_k}")
    print("=" * 80)

    # Retrieve relevant paragraphs from Pinecone
    contexts = get_context(question, top_k=top_k)

    # Extract and print answers from the retrieved paragraphs
    extract_answer(question, contexts)


QUESTION: Who was the first person to walk on the Moon?
TOP K: 1
[{'answer': 'Armstrong',
  'context': 'The trip to the Moon took just over three days. After achieving '
             'orbit, Armstrong and Aldrin transferred into the Lunar Module, '
             'named Eagle, and after a landing gear inspection by Collins '
             'remaining in the Command/Service Module Columbia, began their '
             'descent. After overcoming several computer overload alarms '
             'caused by an antenna switch left in the wrong position, and a '
             'slight downrange error, Armstrong took over manual flight '
             'control at about 180 meters (590 ft), and guided the Lunar '
             'Module to a safe landing spot at 20:18:04 UTC, July 20, 1969 '
             '(3:17:04 pm CDT). The first humans on the Moon would wait '
             'another six hours before they ventured out of their craft. At '
             '02:56 UTC, July 21 (9:56 pm CDT July 20), Armstrong

The retriever did not always return a paragraph containing the answer. For the Moon question, top_k=1 returned a relevant paragraph and the reader correctly extracted “Armstrong.” With top_k=3, it also produced two incorrect answers from other paragraphs.

The Einstein question returned paragraphs that did not state which university he attended, so none of the extracted answers answered the question. Their confidence scores were very low.

For the Mars hotel question, the system retrieved paragraphs about hotels in Atlantic City and returned hotel names, even though those paragraphs did not answer the question.

These results show that retrieving more paragraphs can provide useful context, but it can also introduce irrelevant results. The reader’s score helps compare its predictions, but it does not guarantee that an answer is correct. The returned context should also be checked.

<br>

---

### Clean Up

Now that we're done experimenting, let's delete the Pinecone index we created. This frees up the resources it was using (and, when using a paid plan, avoids being billed for an index we no longer need).

In [ ]:
# To delete the Pinecone index,
# uncomment the line below + run this code cell

# pc.delete_index(index_name)